In [2]:
# =============================================================================
# SETUP
# =============================================================================
import polars as pl
from utils.constants import get_open_data_dirs, get_j1_league_dirs

J1_LEAGUE_DIR = get_j1_league_dirs()
OPEN_DATA_DIR = get_open_data_dirs()


# Loads ALL parquets in the folder as a single dataframe, lazy (doesn’t use memory until you need it)
df = pl.scan_parquet(OPEN_DATA_DIR["bronze_events"], extra_columns="ignore", missing_columns="insert")


In [3]:
df_head = df.head(5).collect()   # For first 5 rows as an eager DataFrame
print(df_head)

df_collected = df.collect()
print(df_collected.shape)
print(df_collected.head(5))
print(df_collected.describe())

NameError: name 'df' is not defined

In [1]:
# Install once if you don't have it
# !pip install -q psutil

import os, time, psutil, polars as pl
from utils.constants import get_open_data_dirs

OPEN_DATA_DIR = get_open_data_dirs()
bronze_dir    = OPEN_DATA_DIR["bronze_events"]  # this already works for you

proc   = psutil.Process(os.getpid())
to_gb  = lambda b: b / 1_073_741_824  # bytes → GiB

def log(label):
    print(f"{label:<20}: {to_gb(proc.memory_info().rss):5.2f} GB")

log("Kernel start")

# Build the lazy plan (no data loaded yet)
lazy_df = pl.scan_parquet(
    f"{bronze_dir}/*.parquet",
    extra_columns="ignore",
    missing_columns="insert"
)
log("After plan")

# Materialise the DataFrame
df = lazy_df.collect()
time.sleep(0.3)              # let the OS update RSS
log("After collect")


Kernel start        :  0.09 GB
After plan          :  0.09 GB
After collect       :  7.12 GB


In [1]:
# !pip install -q psutil pandas pyarrow  # install if not already

import os, psutil, time, pandas as pd
from pathlib import Path
from utils.constants import get_open_data_dirs

proc   = psutil.Process(os.getpid())
gb     = lambda b: b / 1_073_741_824
log    = lambda lbl: print(f"{lbl:<25}: {gb(proc.memory_info().rss):5.2f} GB")

OPEN_DATA_DIR = get_open_data_dirs()
bronze_dir    = Path(OPEN_DATA_DIR["bronze_events"]).resolve()

files = list(bronze_dir.glob("*.parquet"))
print(f"{len(files)} Parquet files found")

log("Kernel start")

# ——— Pandas read + concat ———
dfs = [pd.read_parquet(f, engine="pyarrow") for f in files]  # engine=pyarrow = fastest
dfp = pd.concat(dfs, ignore_index=True)

time.sleep(0.3)   # let OS update RSS
log("After pandas concat")      # full Pandas DataFrame now in RAM

print("Pandas shape:", dfp.shape)


3433 Parquet files found
Kernel start             :  0.13 GB
After pandas concat      : 13.76 GB
Pandas shape: (12083338, 149)
